# Insurance Fraud Detection - Oracle Dataset
**Dataset:** fraud_oracle.csv (15,420 records)
**Target:** FraudFound_P (0/1)
**Task:** Binary Classification

## Activity 1: Collect the Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
import joblib, warnings
warnings.filterwarnings('ignore')
print('Libraries imported')

In [ ]:
df = pd.read_csv('fraud_oracle.csv')
print('Shape:', df.shape)
df.head()

## Activity 2: Data Preparation

In [ ]:
df.info()

In [ ]:
print('Missing values:', df.isnull().sum().sum())
print('Target:\n', df['FraudFound_P'].value_counts())
print(f'Fraud rate: {df["FraudFound_P"].mean()*100:.2f}%')

In [ ]:
df.drop(columns=['PolicyNumber','RepNumber'], inplace=True)
print('Shape after drop:', df.shape)

## Story 2: Visual Analysis

In [ ]:
plt.figure(figsize=(7,4))
ax = df['FraudFound_P'].value_counts().plot(kind='bar', color=['#00897B','#D63031'], edgecolor='white')
for p in ax.patches:
    ax.annotate(str(int(p.get_height())), (p.get_x()+p.get_width()/2, p.get_height()+50), ha='center', fontweight='bold')
plt.title('Fraud vs Legitimate Claims', fontweight='bold')
plt.xticks([0,1], ['Legitimate','Fraud'], rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
df.groupby(['AccidentArea','FraudFound_P']).size().unstack().plot(kind='bar', color=['#00897B','#D63031'])
plt.title('Accident Area vs Fraud')
plt.xticks(rotation=0)
plt.legend(['Legitimate','Fraud'])
plt.tight_layout()
plt.show()

In [ ]:
fig,axes = plt.subplots(1,2,figsize=(12,4))
for col,ax in zip(['PoliceReportFiled','WitnessPresent'],axes):
    df.groupby([col,'FraudFound_P']).size().unstack().fillna(0).plot(kind='bar',ax=ax,color=['#00897B','#D63031'])
    ax.set_title(col)
    ax.set_xticklabels(ax.get_xticklabels(),rotation=0)
plt.tight_layout()
plt.show()

## Story 2.3: Multivariate Analysis

In [ ]:
df_enc = df.copy()
le_tmp = LabelEncoder()
for col in df_enc.select_dtypes(include='object').columns:
    df_enc[col] = le_tmp.fit_transform(df_enc[col].astype(str))
corr = df_enc.corr()['FraudFound_P'].abs().sort_values(ascending=False).head(15)
plt.figure(figsize=(10,6))
corr.drop('FraudFound_P').plot(kind='barh', color='#0D1B2A')
plt.title('Top Feature Correlations with Fraud', fontweight='bold')
plt.tight_layout()
plt.show()

## Encoding Categorical Features

In [ ]:
le_dict = {}
for col in df.select_dtypes(include='object').columns:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    le_dict[col] = le
print('Encoding done. Shape:', df.shape)
df.head()

## Story: Scaling

In [ ]:
X = df.drop('FraudFound_P', axis=1)
y = df['FraudFound_P']
feature_names = X.columns.tolist()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42, stratify=y)
print('Train:', X_train.shape, '| Test:', X_test.shape)

## Story 1.1: Decision Tree

In [ ]:
dt = DecisionTreeClassifier(max_depth=5, random_state=42)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)
acc_dt = accuracy_score(y_test, y_pred_dt)
print(f'Decision Tree: {acc_dt:.4f}')
print(classification_report(y_test, y_pred_dt, target_names=['Legitimate','Fraud']))

## Story 1.2: Random Forest

In [ ]:
rf = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
acc_rf = accuracy_score(y_test, y_pred_rf)
print(f'Random Forest: {acc_rf:.4f}')
print(classification_report(y_test, y_pred_rf, target_names=['Legitimate','Fraud']))

## Activity 1.3: KNN

In [ ]:
knn = KNeighborsClassifier(n_neighbors=7)
knn.fit(X_train, y_train)
y_pred_knn = knn.predict(X_test)
acc_knn = accuracy_score(y_test, y_pred_knn)
print(f'KNN: {acc_knn:.4f}')
print(classification_report(y_test, y_pred_knn, target_names=['Legitimate','Fraud']))

## Story 1.4: Logistic Regression

In [ ]:
lr = LogisticRegression(max_iter=500, random_state=42)
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)
acc_lr = accuracy_score(y_test, y_pred_lr)
print(f'Logistic Regression: {acc_lr:.4f}')
print(classification_report(y_test, y_pred_lr, target_names=['Legitimate','Fraud']))

## Story 1.5: Naive Bayes

In [ ]:
nb_clf = GaussianNB()
nb_clf.fit(X_train, y_train)
y_pred_nb = nb_clf.predict(X_test)
acc_nb = accuracy_score(y_test, y_pred_nb)
print(f'Naive Bayes: {acc_nb:.4f}')
print(classification_report(y_test, y_pred_nb, target_names=['Legitimate','Fraud']))

## Story 1.6: SVM

In [ ]:
svm = SVC(kernel='rbf', probability=True, random_state=42)
svm.fit(X_train, y_train)
y_pred_svm = svm.predict(X_test)
acc_svm = accuracy_score(y_test, y_pred_svm)
print(f'SVM: {acc_svm:.4f}')
print(classification_report(y_test, y_pred_svm, target_names=['Legitimate','Fraud']))

## Story 1.7: Confusion Matrices

In [ ]:
fig,axes = plt.subplots(2,3,figsize=(15,10))
axes = axes.flatten()
for idx,(name,preds) in enumerate([('Decision Tree',y_pred_dt),('Random Forest',y_pred_rf),('KNN',y_pred_knn),('Logistic Regression',y_pred_lr),('Naive Bayes',y_pred_nb),('SVM',y_pred_svm)]):
    cm = confusion_matrix(y_test,preds)
    sns.heatmap(cm,annot=True,fmt='d',ax=axes[idx],cmap='Blues',xticklabels=['Legit','Fraud'],yticklabels=['Legit','Fraud'])
    axes[idx].set_title(f'{name}\n{accuracy_score(y_test,preds):.2%}', fontweight='bold')
plt.suptitle('Confusion Matrices', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## Compare the Models

In [ ]:
results = {'Decision Tree':acc_dt,'Random Forest':acc_rf,'KNN':acc_knn,'Logistic Regression':acc_lr,'Naive Bayes':acc_nb,'SVM':acc_svm}
results_df = pd.DataFrame(list(results.items()),columns=['Model','Accuracy']).sort_values('Accuracy',ascending=False).reset_index(drop=True)
results_df['Accuracy_%'] = (results_df['Accuracy']*100).round(2)
print(results_df.to_string(index=False))
print(f'Best: {results_df.iloc[0]["Model"]} @ {results_df.iloc[0]["Accuracy_%"]}%')

In [ ]:
plt.figure(figsize=(10,5))
colors = ['#D63031' if i==0 else '#0D1B2A' for i in range(len(results_df))]
bars = plt.barh(results_df['Model'], results_df['Accuracy_%'], color=colors)
for bar,val in zip(bars,results_df['Accuracy_%']):
    plt.text(bar.get_width()+0.1, bar.get_y()+bar.get_height()/2, f'{val}%', va='center', fontweight='bold')
plt.title('Model Accuracy Comparison', fontweight='bold')
plt.xlabel('Accuracy (%)')
plt.xlim(78,97)
plt.tight_layout()
plt.show()

## Save Best Model

In [ ]:
import os
os.makedirs('models', exist_ok=True)
best_name = results_df.iloc[0]['Model']
all_models = {'Decision Tree':dt,'Random Forest':rf,'KNN':knn,'Logistic Regression':lr,'Naive Bayes':nb_clf,'SVM':svm}
best_mdl = all_models[best_name]
joblib.dump(best_mdl, 'models/best_model.pkl')
joblib.dump(scaler, 'models/scaler.pkl')
joblib.dump(le_dict, 'models/label_encoders.pkl')
joblib.dump(feature_names, 'models/feature_names.pkl')
joblib.dump(results, 'models/model_results.pkl')
joblib.dump(best_name, 'models/best_model_name.pkl')
print(f'Saved: {best_name} ({results[best_name]*100:.2f}%)')
print('Run: python app.py  ->  http://localhost:5000')